# 🏯 템플릿 2 — 텍스트 RPG (엔딩 있는 스토리)

LLM이 게임 마스터가 되어 학생만의 세계관에서 RPG를 진행. 턴 수에 따라 자연스럽게 엔딩으로 이어집니다.

## 핵심 구조
- **세계관 + 캐릭터**: 시스템 프롬프트에 정의
- **상태 관리**: 턴 수, HP, 인벤토리를 Python에서 추적
- **엔딩 시스템**: 마지막 턴 도달 시 LLM이 후일담 생성

## 학생이 가장 쉽게 손댈 곳
- `WORLD` 변수: 본인의 세계관 (조선, 우주, 학교, 좀비, K-pop 데뷔 등)
- `ENDINGS` 리스트: 가능한 엔딩들
- `MAX_TURNS`: 게임 길이

In [ ]:
!pip install -q gradio openai

In [ ]:
# ===== Part 0. 서버 연결 =====
# 강사가 안내한 5090 cloudflared URL
SERVER_URL = "https://YOUR_URL.trycloudflare.com".strip().rstrip("/")
MODEL = "qwen2.5:7b-instruct"

assert "YOUR" not in SERVER_URL, "❌ SERVER_URL을 강사가 알려준 URL로 바꾸세요"

import httpx
from openai import OpenAI

client = OpenAI(
    base_url=f"{SERVER_URL}/v1",
    api_key="ollama",
    http_client=httpx.Client(headers={"User-Agent": "Mozilla/5.0"}),
)
print("✅ LLM 서버 연결 OK")

In [ ]:
# ===== 학생이 본인 게임으로 바꿀 부분 =====

WORLD = {
    "title": "🏯 잃어버린 보물의 던전",
    "setting": """당신은 전설로만 전해지던 고대 던전을 탐험하는 모험가입니다.
던전 깊은 곳엔 황금 왕관이 잠들어 있다고 합니다.
하지만 함정과 수수께끼, 그리고 무언가 살아있는 것들이 당신을 기다리고 있습니다.""",
    "start": "당신은 횃불 하나를 들고 던전 입구 앞에 섰습니다. 차가운 공기가 안에서 흘러나옵니다.",
}

ENDINGS = [
    "황금 왕관 획득 (영웅의 귀환)",
    "함정에 빠져 사망 (비극)",
    "현자와의 만남 (깨달음의 길)",
    "괴물과 친구가 됨 (반전 엔딩)",
    "다른 차원으로 이동 (수수께끼의 결말)",
]

MAX_TURNS = 8       # ← 게임 길이. 짧을수록 빨리 끝남
START_HP = 100
START_INVENTORY = ["횃불", "단검", "물 1병"]

# 다른 세계관 예시 (위 WORLD를 통째로 바꿔보세요):
# 🎤 "K-pop 데뷔 도전기" - 연습생 → 메이저 데뷔 or 잘림
# 🚀 "우주 표류기" - 고장난 우주선에서 탈출
# 🎓 "수능 D-30" - 마지막 한 달 동안의 선택
# 🧟 "좀비 학교" - 좀비 사태 속 학교에서 생존
# 🕵️ "탐정 사무소" - 의뢰 사건 해결

In [ ]:
SYSTEM_PROMPT_TEMPLATE = """당신은 텍스트 RPG의 게임 마스터입니다. 한국어로만 답하세요.

[세계관]
{setting}

[규칙]
- 매 턴 짧고 생생하게 묘사 (3-5문장)
- 마지막에 플레이어가 할 수 있는 행동 선택지 2-3개 제시 (번호 매기기)
- 플레이어 결정에 따라 HP, 인벤토리가 자연스럽게 변할 수 있음
- 분위기를 점점 고조시켜 클라이맥스로
- 위험/긴장감 / 발견 / 인물 등장을 적절히 섞기

[가능한 엔딩]
{endings}

[현재 상태]
- 턴: {turn}/{max_turns}
- HP: {hp}
- 인벤토리: {inventory}
"""

def build_system(state):
    return SYSTEM_PROMPT_TEMPLATE.format(
        setting=WORLD["setting"],
        endings="\n".join(f"- {e}" for e in ENDINGS),
        turn=state["turn"],
        max_turns=MAX_TURNS,
        hp=state["hp"],
        inventory=", ".join(state["inventory"]),
    )

def init_state():
    return {
        "turn": 0,
        "hp": START_HP,
        "inventory": list(START_INVENTORY),
        "history": [],
        "ended": False,
    }

def play(message, history, state):
    if state is None or state.get("ended"):
        return history + [(message, "🔄 '게임 새로 시작' 버튼을 눌러 새 게임을 시작하세요")], state

    state["turn"] += 1
    is_final = state["turn"] >= MAX_TURNS

    state["history"].append({"role": "user", "content": message})

    sys = build_system(state)
    if is_final:
        sys += f"""

[중요] 이번이 마지막 턴입니다. 지금까지의 흐름을 종합해서 
**"### 🎬 엔딩: [엔딩 제목]"** 형식으로 시작하고
주인공의 후일담을 2-3 문단으로 들려주며 게임을 마무리하세요."""

    messages = [{"role": "system", "content": sys}] + state["history"][-8:]

    resp = client.chat.completions.create(
        model=MODEL, messages=messages,
        max_tokens=500, temperature=0.85,
    )
    reply = resp.choices[0].message.content
    state["history"].append({"role": "assistant", "content": reply})

    if is_final or "엔딩:" in reply:
        state["ended"] = True

    status = f"턴 **{state['turn']}/{MAX_TURNS}** | HP `{state['hp']}` | 🎒 {', '.join(state['inventory'])}"
    if state["ended"]:
        status += " | 🏁 **게임 종료**"

    return history + [(message, reply)], state, status

def reset():
    state = init_state()
    intro = WORLD["start"] + "\n\n어떻게 하시겠습니까?\n1. 안으로 들어간다\n2. 주변을 먼저 살펴본다\n3. 입구에서 기다린다"
    history = [("게임 시작", intro)]
    state["history"] = [
        {"role": "user", "content": "게임 시작"},
        {"role": "assistant", "content": intro},
    ]
    status = f"턴 **0/{MAX_TURNS}** | HP `{START_HP}` | 🎒 {', '.join(START_INVENTORY)}"
    return history, state, status

# UI
with gr.Blocks(title=WORLD["title"], theme=gr.themes.Soft()) as demo:
    gr.Markdown(f"# {WORLD['title']}")
    state = gr.State()
    status = gr.Markdown(f"턴 **0/{MAX_TURNS}** | HP `{START_HP}` | 🎒 {', '.join(START_INVENTORY)}")
    chat = gr.Chatbot(label="모험 일지", height=450)
    with gr.Row():
        msg = gr.Textbox(label="행동", placeholder="예: 검을 빼들고 천천히 다가간다", scale=4)
        send = gr.Button("➤ 행동", variant="primary", scale=1)
    reset_btn = gr.Button("🔄 게임 새로 시작")

    send.click(play, [msg, chat, state], [chat, state, status]).then(lambda: "", outputs=msg)
    msg.submit(play, [msg, chat, state], [chat, state, status]).then(lambda: "", outputs=msg)
    reset_btn.click(reset, outputs=[chat, state, status])

demo.launch(share=True)

---
## 🚀 바이브 코딩 확장 아이디어

### 쉬움
- 세계관 통째로 바꾸기 (좀비, K-pop, 수능, 데이트 시뮬)
- 엔딩 더 많이 추가
- HP/인벤토리가 진짜로 변하도록 (LLM 출력 파싱)

### 중간
- 캐릭터 시트 입력란 (이름, 직업, 능력치)
- 주사위 굴리기 시스템 추가 (성공/실패 판정)
- 명예 점수, 운명 카드 등 게임 메커닉
- 5회차 이미지 생성 API 호출해서 매 턴 장면 일러스트

### 도전적
- 멀티 엔딩 분기 트리 (저장+불러오기)
- BGM 추가 (분위기에 맞춰 자동 선택)
- 다른 친구가 NPC로 참여
- 4회차 TTS 붙여서 음성으로 narration

### 🎁 자랑하기 팁
- 본인 일상을 RPG화 ("취준생 생존기", "장교후보생 24개월")
- 친구를 NPC로 등장시켜서 URL 공유하면 다들 좋아함